# EDA - astronomy caption dataset

**EDA only.** No training, no splitting, no evaluation logic in this notebook.
Anything that affects results lives in `src/` so that execution order cannot
change the outcome - that is exactly how the original project ended up training
on its own test set.

Run `make data features` first.


In [ ]:
import sys
from pathlib import Path

# so `import src...` works when the notebook is opened from notebooks/
sys.path.insert(0, str(Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()))

import pandas as pd

from src.config import load_config
from src.data.build import load_processed
from src.data.download import load_raw
from src.data.split import load_manifest
from src.data.validate import validate_records

cfg = load_config()
cfg.paths.raw_dir


## Raw vs processed


In [ ]:
raw = load_raw(cfg)
processed = load_processed(cfg)
print(f'raw {len(raw)} -> processed {len(processed)}  (clean_mode={cfg.features.clean_mode})')
df = pd.DataFrame(processed)
df.head()


## Validation report


In [ ]:
report = validate_records(processed, cfg.paths.raw_dir)
{k: v for k, v in report.items() if not isinstance(v, list)}


## Caption length


In [ ]:
df['chars'] = df['text'].str.len()
df['words'] = df['text'].str.split().str.len()
display(df[['chars', 'words']].describe().T)
ax = df['words'].plot.hist(bins=20, edgecolor='black', figsize=(8, 4))
ax.set_xlabel('words'); ax.set_title('Caption word count');


## Label balance

The heuristic label is only used for stratified splitting and this plot.
It is **not** a training target. See `MODEL_CARD.md` on the imbalance.


In [ ]:
counts = df['label'].value_counts()
display(counts)
ax = counts.plot.bar(edgecolor='black', figsize=(8, 4))
ax.set_ylabel('captions'); ax.set_title('Heuristic label distribution');


## Split sanity check

Read-only: the notebook never creates a split, it only inspects the one
`src/data/split.py` already wrote.


In [ ]:
manifest = load_manifest(cfg)
print('split_hash:', manifest['split_hash'])
print('counts    :', manifest['counts'])

train, val, test = (set(manifest['ids'][k]) for k in ('train', 'val', 'test'))
print('train n val :', len(train & val))
print('train n test:', len(train & test))
print('val   n test:', len(val & test))
assert not (train & val) and not (train & test) and not (val & test)
print('
held-out is clean')


## Label balance per split

Small held-out sets can miss a whole class - worth looking at before reading
any aggregate metric.


In [ ]:
by_id = {r['image_id']: r for r in processed}
rows = [
    {'split': name, 'label': by_id[i]['label']}
    for name in ('train', 'val', 'test')
    for i in manifest['ids'][name]
]
pd.crosstab(pd.DataFrame(rows)['label'], pd.DataFrame(rows)['split'])
